# CKP02 — AnalistaBot / DocMind RAG

**FIAP · 2026.2 · Módulo 2**  
**Domínio:** análise de ações e mercado de capitais.  
**Autor:** Miguel Piedade (572445).

**Objetivo:** recuperar evidência de cinco publicações oficiais da CVM, comparar `256/32` e `512/64` e medir `faithfulness` e `answer_relevancy` reais com RAGAS.



## 1. Enviar o ZIP do projeto


In [52]:
import os, sys, zipfile
from pathlib import Path
EM_COLAB = "google.colab" in sys.modules or bool(os.getenv("COLAB_RELEASE_TAG"))
if EM_COLAB:
    from google.colab import files
    enviados = files.upload()
    zip_path = next((Path(nome) for nome in enviados if nome.endswith(".zip")), None)
    if zip_path is None:
        raise FileNotFoundError("Envie o ZIP do projeto")
    with zipfile.ZipFile(zip_path) as arquivo:
        destino = Path("/content").resolve()
        for item in arquivo.infolist():
            if not (destino / item.filename).resolve().is_relative_to(destino):
                raise ValueError("Caminho inválido no ZIP")
        arquivo.extractall(destino)
    raiz = destino / "AnalistaBot_CKP02"
else:
    raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "app/config.py").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("Abra o notebook dentro do projeto")
os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Projeto:", raiz.name)


Saving AnalistaBot_CKP02_entrega.zip to AnalistaBot_CKP02_entrega.zip
Projeto: AnalistaBot_CKP02


## 2. Instalar as dependências


In [53]:

import subprocess

if EM_COLAB:
    # Dependências do sistema necessárias para o Ollama
    subprocess.check_call([
        "apt-get", "update", "-qq"
    ])

    subprocess.check_call([
        "apt-get", "install", "-y", "-qq", "zstd"
    ])

    # Dependências Python do projeto
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "-r", "requirements.txt"
    ])
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "check"
    ])

print("Dependências verificadas")


Dependências verificadas


In [54]:
import subprocess

if EM_COLAB:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "zstd"])

    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "-r", "requirements.txt"
    ])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "check"])

print("Dependências verificadas")

Dependências verificadas


## 3. Ollama Cloud



In [55]:
import os

if EM_COLAB:
    from google.colab import userdata

    os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY").strip()
    os.environ["OLLAMA_CHAT_BASE_URL"] = "https://ollama.com"
    os.environ["OLLAMA_EMBED_BASE_URL"] = "http://localhost:11434"

from app.chain import build_llm

# Evita reutilizar um cliente com credenciais antigas
build_llm.cache_clear()

llm = build_llm()

print("Modelo remoto:", llm.model)
print("Autenticação configurada sem exposição da chave")

Modelo remoto: gemma4:31b
Autenticação configurada sem exposição da chave


## 4. Ollama local somente para `nomic-embed-text`


In [56]:
import shutil, time, requests
if EM_COLAB:
    if shutil.which("ollama") is None:
        subprocess.check_call("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
    except requests.RequestException:
        servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for tentativa in range(30):
            try:
                requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
                break
            except requests.RequestException:
                time.sleep(2)
        else:
            raise RuntimeError("Ollama não iniciou")
    subprocess.check_call(["ollama", "pull", "nomic-embed-text"])
from app.rag import build_embeddings
print("Dimensão embed_query:", len(build_embeddings().embed_query("mercado de ações")))
print("Vetores embed_documents:", len(build_embeddings().embed_documents(["Educação financeira"])))

Dimensão embed_query: 768
Vetores embed_documents: 1


## 5. Base real e links de origem

- [Caderno CVM 15: Day Trade](https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/cadernos/caderno-cvm-15-day_trade.pdf/@@display-file/file) — 2020.
- [Caderno CVM 14: BDR - Brazilian Depositary Receipts](https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/cadernos/cvm-caderno-14.pdf/@@display-file/file) — 2020.
- [Caderno CVM 13: Mecanismo de Ressarcimento de Prejuízos](https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/cadernos/cvm-caderno-13.pdf/@@display-file/file) — 2020.
- [Guia CVM: Suitability](https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/guias/guia-de-suitability-lancamento/@@display-file/file) — 2022.
- [CVM: Investidor 60+](https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/guias/investidor-60.pdf/@@display-file/file) — 2026.

Os cinco PDFs originais estão incluídos em `data/documentos/`.


In [57]:
subprocess.check_call([sys.executable, "-m", "scripts.baixar_fontes"])
from app.documentos import carregar_documentos
from app.config import DOCUMENTOS, CHUNK_SIZES, chunk_config
from pathlib import Path
print("Arquivos:", [p.name for p in DOCUMENTOS.glob("*.pdf")])
documentos = carregar_documentos(minimo=5)
print("Páginas textuais:", len(documentos))
print("Configurações:", [chunk_config(t) for t in CHUNK_SIZES])

Arquivos: ['02_CVM_BDR_2020.pdf', '05_CVM_Investidor_60_2026.pdf', '04_CVM_Suitability_2022.pdf', '03_CVM_MRP_2020.pdf', '01_CVM_Day_Trade_2020.pdf']
Páginas textuais: 152
Configurações: [{'chunk_size': 256, 'chunk_overlap': 32, 'separators': ['\n\n', '\n', '. ', ' ', '']}, {'chunk_size': 512, 'chunk_overlap': 64, 'separators': ['\n\n', '\n', '. ', ' ', '']}]


## 6. Embed → Chroma persistente (duas coleções)


In [58]:
from scripts.indexar import main as indexar_base
indexar_base()
import chromadb
from app.config import CHROMA_DIR
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
print("Coleções:", [(c.name, c.count()) for c in client.list_collections()])


Teste embed_query(): 768 dimensões
Teste embed_documents(): 1 vetor
{'colecao': 'analise_de_acoes_chunk_256', 'documentos': 5, 'chunks': 835, 'cache': True}
{'colecao': 'analise_de_acoes_chunk_512', 'documentos': 5, 'chunks': 448, 'cache': True}
Coleções: [('analise_de_acoes_chunk_256', 835), ('analise_de_acoes_chunk_512', 448)]


## 7. Demonstração Retrieve → Generate com fonte/página/chunk


In [59]:
from app.rag import responder, buscar
pergunta_demo = "O que é um BDR segundo a CVM?"
res = responder(pergunta_demo, chunk_size=512)
print("Pergunta:", pergunta_demo)
print("Resposta:", res["resposta"])
print("Fontes:", [{"chunk": f["chunk_id"], "arquivo": f["arquivo"], "pagina": f["pagina"], "url": f["fonte"]} for f in res["fontes"]])

Pergunta: O que é um BDR segundo a CVM?
Resposta: Com base nos documentos fornecidos, os BDRs (Brazilian Depositary Receipts) são certificados de depósito cujos programas, para serem emitidos e negociados no Brasil, devem obrigatoriamente ser objeto de registro na Comissão de Valores Mobiliários (CVM) [512_000191].

A emissão desses certificados é classificada em programas, que podem ser patrocinados ou não pelo emissor dos valores mobiliários que servem de objeto ao certificado [512_000191]. Esses programas diferem entre si conforme as características de negociação, distribuição e divulgação de informações [512_000191].

*Lembre-se: este conteúdo é educativo e não constitui recomendação de investimento.*
Fontes: [{'chunk': '512_000197', 'arquivo': '02_CVM_BDR_2020.pdf', 'pagina': 15, 'url': 'https://www.gov.br/investidor/pt-br/educacional/publicacoes-educacionais/cadernos/cvm-caderno-14.pdf/@@display-file/file'}, {'chunk': '512_000191', 'arquivo': '02_CVM_BDR_2020.pdf', 'pagina': 13, 

## 8. RAGAS: 6 perguntas × 2 chunkings


In [60]:
from app.evaluation import avaliar_configuracoes
tabela, medias, vencedor = avaliar_configuracoes()
from IPython.display import display
print("=== RAGAS por pergunta e configuração ===")
display(tabela[["chunk_size", "user_input", "faithfulness", "answer_relevancy"]])
print("=== RAGAS: médias ===")
display(medias)
print("Vencedor pelo critério definido (faithfulness, desempate relevancy):", vencedor)
print((Path("resultados") / "resumo_avaliacao.md").read_text(encoding="utf-8"))

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

=== RAGAS por pergunta e configuração ===


,chunk_size,user_input,faithfulness,answer_relevancy
0,256,Como o caderno da CVM define day trade e qual ...,0.857143,0.871036
1,256,O que é um BDR e como esse recibo se relaciona...,0.857143,0.770092
2,256,Qual é a finalidade do Mecanismo de Ressarcime...,1.000000,0.871738
3,256,O que significa suitability e por que é import...,0.666667,0.922136
4,256,Quais cuidados o guia Investidor 60+ recomenda...,0.800000,1.000000
5,256,"Segundo o material sobre day trade, como análi...",0.875000,0.788045
6,512,Como o caderno da CVM define day trade e qual ...,1.000000,0.861912
7,512,O que é um BDR e como esse recibo se relaciona...,0.800000,0.874835
8,512,Qual é a finalidade do Mecanismo de Ressarcime...,1.000000,0.928793
9,512,O que significa suitability e por que é import...,0.875000,0.909202


=== RAGAS: médias ===


,chunk_size,faithfulness,answer_relevancy
1,512,0.922024,0.897566
0,256,0.842659,0.870508


Vencedor pelo critério definido (faithfulness, desempate relevancy): 512
# Comparação RAGAS medida

- Melhor chunk_size (critério: faithfulness médio; desempate por answer_relevancy): **512**.
- Média de faithfulness: **0.922**; answer_relevancy: **0.898**.
- Meta >=0,7: **atingida**.
- Cautela: resultados medidos pelo RAGAS, não garantias de factualidade.
- Faça revisão humana das respostas e dos trechos citados antes da entrega.



## 9. Evidências e entrega


In [61]:
from IPython.display import FileLink, display
for caminho in sorted(Path("resultados").glob("*")):
    if caminho.is_file():
        display(FileLink(str(caminho)))
print("Total de PDFs reais:", len(list(Path("data/documentos").glob("*.pdf"))))

/content/AnalistaBot_CKP02/resultados/.gitkeep

/content/AnalistaBot_CKP02/resultados/configuracao_vencedora.json

/content/AnalistaBot_CKP02/resultados/demonstracao_memoria.json

/content/AnalistaBot_CKP02/resultados/evidencias_respostas.json

/content/AnalistaBot_CKP02/resultados/ragas_chunk_256.csv

/content/AnalistaBot_CKP02/resultados/ragas_chunk_512.csv

/content/AnalistaBot_CKP02/resultados/ragas_comparacao.csv

/content/AnalistaBot_CKP02/resultados/ragas_medias.csv

/content/AnalistaBot_CKP02/resultados/resumo_avaliacao.md

/content/AnalistaBot_CKP02/resultados/verificacao_entrega.json

Total de PDFs reais: 5
